# CLP Graph Time Series: Step-by-Step Anatomy (Energy Data)

A guided, exhaustive walkthrough of the `graph_Time_series` CLP framework
using **real electricity consumption data** from the GiftEval benchmark.

This notebook focuses on:
1. Loading and inspecting real energy time-series data.
2. Detailed token-by-token replay of a CLP sequence, with rich state descriptions at every step.
3. A complete reference of every token's **inputs**, **outputs**, and **grammar adjacency** (which tokens can precede/follow it).


## 0. Setup
Install dependencies and set up import paths.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()

def run(cmd):
    print("$", " ".join(cmd))
    subprocess.check_call(cmd)

if IN_COLAB:
    os.chdir("/content")
    REPO = Path("/content/graph_Time_series")
    KERNELS_REPO = Path("/content/kernels_playground")

    if REPO.exists():
        run(["git", "-C", str(REPO), "pull", "--ff-only"])
    else:
        run(["git", "clone", "https://github.com/chahineNejm/graph_Time_series.git", str(REPO)])

    if KERNELS_REPO.exists():
        run(["git", "-C", str(KERNELS_REPO), "pull", "--ff-only"])
    else:
        run(["git", "clone", "https://github.com/chahineNejm/kernels_playground.git", str(KERNELS_REPO)])

    req = KERNELS_REPO / "first_tests" / "requirements.txt"
    run([
        sys.executable, "-m", "pip", "install", "-q",
        "-r", str(req),
        "numpy", "scipy", "scikit-learn", "networkx", "matplotlib",
        "tqdm", "datasets", "xgboost", "PyWavelets"
    ])
else:
    HERE = Path.cwd()
    if HERE.name == "graph_Time_series":
        REPO = HERE
    elif HERE.parent.name == "graph_Time_series":
        REPO = HERE.parent
    elif (HERE / "graph_Time_series").exists():
        REPO = HERE / "graph_Time_series"
    else:
        REPO = HERE

    kernel_candidates = [
        REPO.parent / "kernels_playground",
        REPO.parent.parent / "kernels_playground",
    ]
    KERNELS_REPO = next((p for p in kernel_candidates if p.exists()), kernel_candidates[0])

FIRST_TESTS = KERNELS_REPO / "first_tests"
for p in [REPO, REPO.parent, FIRST_TESTS]:
    p = str(p.resolve())
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect
import importlib
import textwrap
from pprint import pprint

import numpy as np
import matplotlib.pyplot as plt

print("IN_COLAB:", IN_COLAB)
print("repo:", REPO, REPO.exists())
print("package dir:", REPO / "graph_Time_series", (REPO / "graph_Time_series").exists())
print("kernels first_tests:", FIRST_TESTS, FIRST_TESTS.exists())


In [ ]:
import graph_Time_series as gts
from graph_Time_series import State, Grammar, plot_grammar, mcts_search, print_mcts_tree
from graph_Time_series.token import Token, _shapes
from graph_Time_series.tokens import register_all
from graph_Time_series.tokens.cleaning import CleanIdentity, CleanDetrend, CleanMovingAvg, CleanNormalize, CleanDetrendNorm
from graph_Time_series.tokens.features import FeatRaw, FeatFFTEncode, FeatLagFeatures
from graph_Time_series.tokens.models import ModelKernelRBF, ModelRandomForest, ModelXGBoost, StopToken, compute_mase
from graph_Time_series.heuristics import compute_mi_score, compute_action_priors

np.set_printoptions(precision=4, suppress=True)


## 1. Load Real Energy Data (electricity\_H\_long)

We load electricity consumption series from the GiftEval benchmark.
These are real hourly metered electricity readings — the kind of signal
the CLP framework is designed to forecast.


In [ ]:
# ── knobs ──────────────────────────────────────────────────────
CONFIG_NAME   = "electricity_H_long"
START, STOP, STEP = 0, 80, 4
HISTORY_LEN   = 2000
FUTURE_LEN    = 720
SEED          = 0

from utils.config import DATASETS
from utils.data import build_examples
from utils.augmentation import uniform_length

raw = build_examples(
    config=CONFIG_NAME,
    start=START,
    stop=STOP,
    step=STEP,
    dataset_name=DATASETS["eval"],
)

fixed = uniform_length(
    raw,
    target_len=HISTORY_LEN,
    min_len=HISTORY_LEN // 2,
    keys=("history",),
    seed=SEED,
    verbose=True,
)

fixed = [r for r in fixed if len(r["future"]) >= FUTURE_LEN]
if len(fixed) < 3:
    raise ValueError("Not enough usable examples. Increase STOP or reduce STEP.")

H = np.stack([np.asarray(r["history"], dtype=np.float32) for r in fixed])
F = np.stack([np.asarray(r["future"][:FUTURE_LEN], dtype=np.float32) for r in fixed])

print(f"Dataset        : {CONFIG_NAME}")
print(f"Raw examples   : {len(raw)}")
print(f"Usable examples: {len(fixed)}")
print(f"H shape        : {H.shape}   (n_samples, history_len)")
print(f"F shape        : {F.shape}   (n_samples, horizon)")


### Quick visual inspection
Plot a single electricity series showing the history window and the future horizon we want to forecast.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))

# Full series
ax = axes[0]
ax.plot(np.arange(-H.shape[1], 0), H[0], label="history", lw=0.8)
ax.plot(np.arange(F.shape[1]), F[0], label="future (ground truth)", lw=0.8)
ax.axvline(0, ls=":", color="grey", alpha=0.6)
ax.set_title("Sample 0 — full view"); ax.legend(); ax.grid(alpha=0.25)
ax.set_xlabel("time step (0 = forecast origin)")

# Zoomed tail
ax = axes[1]
tail = 120
ax.plot(np.arange(-tail, 0), H[0, -tail:], label="history (last 120)", lw=0.8)
ax.plot(np.arange(F.shape[1]), F[0], label="future", lw=0.8)
ax.axvline(0, ls=":", color="grey", alpha=0.6)
ax.set_title("Sample 0 — zoomed near forecast origin"); ax.legend(); ax.grid(alpha=0.25)
ax.set_xlabel("time step")

plt.tight_layout(); plt.show()


### Dataset statistics

In [ ]:
print("Per-sample history statistics:")
print(f"  mean of means : {H.mean(axis=1).mean():.2f}")
print(f"  mean of stds  : {H.std(axis=1).mean():.2f}")
print(f"  global min    : {H.min():.2f}")
print(f"  global max    : {H.max():.2f}")
print()
print("Per-sample future statistics:")
print(f"  mean of means : {F.mean(axis=1).mean():.2f}")
print(f"  mean of stds  : {F.std(axis=1).mean():.2f}")
print(f"  global min    : {F.min():.2f}")
print(f"  global max    : {F.max():.2f}")


## 2. Token Reference — Inputs, Outputs, and Grammar Adjacency

Every token in the CLP framework reads certain feature keys from the state,
writes new feature keys, and has a defined place in the grammar graph
(which tokens may precede it, which may follow it).

This section prints a complete reference card for every registered token.


In [ ]:
g = Grammar()
register_all(g)

# Build predecessor / successor maps from the directed graph
predecessors = {}
successors   = {}
for node in g.graph.nodes:
    predecessors[node] = sorted(g.graph.predecessors(node))
    successors[node]   = sorted(g.graph.successors(node))

print("=" * 90)
print(f"{'TOKEN REFERENCE':^90s}")
print("=" * 90)

for name in sorted(g.tokens.keys()):
    tok = g.tokens[name]
    print(f"\n{'─' * 90}")
    print(f"  Token name    : {tok.name}")
    print(f"  Token class   : {tok.token_class}")
    print(f"  Description   : {tok.description}")
    print()
    print(f"  INPUTS  (reads from state.features):")
    if tok.reads:
        for r in tok.reads:
            print(f"      ← {r}")
    else:
        print(f"      (none — operates on state directly)")
    print()
    print(f"  OUTPUTS (writes to state.features):")
    if tok.writes:
        for w in tok.writes:
            print(f"      → {w}")
    else:
        print(f"      (modifies state metadata / prediction stack)")
    print()
    print(f"  GRAMMAR ADJACENCY:")
    preds = predecessors.get(name, [])
    succs = successors.get(name, [])
    print(f"      Can follow (predecessors) : {preds if preds else ['START (or none)']}")
    print(f"      Can precede (successors)  : {succs if succs else ['(terminal)']}")

print(f"\n{'─' * 90}")
print(f"\nTotal tokens registered: {g.n_tokens}")
print(f"Total grammar edges    : {g.graph.number_of_edges()}")


### Grammar graph visualization

In [ ]:
plot_grammar(g, title="CLP Grammar — valid token transitions")


### Adjacency matrix (who connects to whom)

In [ ]:
all_names = ["START"] + sorted(g.tokens.keys())
n = len(all_names)
idx = {name: i for i, name in enumerate(all_names)}

adj = np.zeros((n, n), dtype=int)
for u, v in g.graph.edges:
    if u in idx and v in idx:
        adj[idx[u], idx[v]] = 1

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(adj, cmap="Blues", aspect="auto")
ax.set_xticks(range(n)); ax.set_xticklabels(all_names, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(n)); ax.set_yticklabels(all_names, fontsize=8)
ax.set_xlabel("To (successor)"); ax.set_ylabel("From (predecessor)")
ax.set_title("Grammar adjacency matrix")
for i in range(n):
    for j in range(n):
        if adj[i, j]:
            ax.text(j, i, "1", ha="center", va="center", fontsize=7, color="white")
plt.tight_layout(); plt.show()


## 3. Detailed Step-by-Step Sequence Replay

This is the core study section. We walk through one CLP token sequence
applied to the **real electricity data**, printing a rich description of
the state after every single token.

For each step you will see:
- What valid actions the grammar allows at this point.
- What the token **reads** from the state and what it **writes**.
- The complete inventory of `state.features` with shapes and summary statistics.
- The prediction stack evolution (cumulative prediction, residual norm).
- Detailed interpretation of what just happened to the data.

**Edit `TOKEN_SEQUENCE` below to study any pipeline you like.**


In [ ]:
TOKEN_SEQUENCE = [
    "normalize",
    "feat_raw",
    "kernel_rbf",
    "STOP",
]


In [ ]:
replay_grammar = Grammar()
register_all(replay_grammar)


def summarize_array(x):
    """Compact statistical summary of an array."""
    x = np.asarray(x)
    if x.size == 0:
        return {"shape": x.shape, "empty": True}
    return {
        "shape": x.shape,
        "mean":  float(np.nanmean(x)),
        "std":   float(np.nanstd(x)),
        "min":   float(np.nanmin(x)),
        "max":   float(np.nanmax(x)),
    }


def describe_state(state, label, token=None):
    """Print a rich, human-readable description of the current state."""
    print(f"\n{'=' * 90}")
    print(f"  STATE AFTER: {label}")
    print(f"{'=' * 90}")

    # ── Basic info ─────────────────────────────────────────────
    seq = " → ".join(state.token_sequence) if state.token_sequence else "(empty — initial state)"
    print(f"  Token chain     : {seq}")
    print(f"  Depth           : {state.depth}")
    print(f"  Last token      : {state.last_token}")
    print(f"  Terminated      : {state.terminated}")
    print(f"  MASE            : {state.mase}")
    print(f"  n_samples       : {state.n_samples}")
    print(f"  horizon         : {state.horizon}")

    # ── What the token did ─────────────────────────────────────
    if token is not None:
        print(f"\n  Token details:")
        print(f"    name        : {token.name}")
        print(f"    class       : {token.token_class}")
        print(f"    description : {token.description}")
        print(f"    reads       : {token.reads}")
        print(f"    writes      : {token.writes}")

    # ── Features inventory ─────────────────────────────────────
    print(f"\n  Features in state ({len(state.features)} keys):")
    for key in sorted(state.features):
        val = state.features[key]
        if hasattr(val, 'shape'):
            s = summarize_array(val)
            print(f"    {key:<28s} shape={str(s['shape']):<18s} "
                  f"mean={s['mean']:>10.4f}  std={s['std']:>10.4f}  "
                  f"min={s['min']:>10.4f}  max={s['max']:>10.4f}")
        else:
            print(f"    {key:<28s} type={type(val).__name__:<12s} value={val}")

    # ── Metadata ───────────────────────────────────────────────
    if state.metadata:
        print(f"\n  Metadata:")
        for k, v in state.metadata.items():
            if hasattr(v, 'shape'):
                print(f"    {k:<28s} shape={v.shape}")
            else:
                print(f"    {k:<28s} = {v}")

    # ── Prediction stack ───────────────────────────────────────
    print(f"\n  Prediction stack ({len(state.prediction_stack)} entries):")
    if state.prediction_stack:
        for i, (pred, pname) in enumerate(zip(state.prediction_stack, state.prediction_names)):
            s = summarize_array(pred)
            print(f"    [{i}] {pname:<16s} shape={str(s['shape']):<18s} "
                  f"mean={s['mean']:>10.4f}  std={s['std']:>10.4f}")
        cum = state.cumulative_prediction()
        print(f"    cumulative        shape={str(cum.shape):<18s} "
              f"mean={np.nanmean(cum):>10.4f}  std={np.nanstd(cum):>10.4f}")
    else:
        print(f"    (empty — no model has run yet)")

    # ── Current target / residual ──────────────────────────────
    tgt = summarize_array(state.current_target)
    print(f"\n  Current target (residual for next model):")
    print(f"    shape={str(tgt['shape']):<18s} "
          f"mean={tgt['mean']:>10.4f}  std={tgt['std']:>10.4f}  "
          f"norm(L2)={np.linalg.norm(state.current_target):>10.4f}")

    # ── Interpretation ─────────────────────────────────────────
    print()


def replay_detailed(sequence, H, F, grammar):
    """
    Run a token sequence on real data with rich state descriptions at every step.
    """
    state = State(H, F)
    describe_state(state, "INITIAL STATE (before any tokens)")

    for step, name in enumerate(sequence, start=1):
        valid = grammar.valid_actions(state)
        print(f"\n{'─' * 90}")
        print(f"  STEP {step}: Applying token '{name}'")
        print(f"  Valid actions at this point: {valid}")
        if name not in valid:
            print(f"  ⚠ WARNING: '{name}' is NOT in the valid set — grammar rules would block this.")
        print(f"{'─' * 90}")

        token = grammar.tokens[name]

        # Show what will be read
        print(f"\n  About to READ from state.features:")
        for r in token.reads:
            if r in state.features:
                val = state.features[r]
                if hasattr(val, 'shape'):
                    print(f"    ← {r:<24s} shape={val.shape}  mean={np.nanmean(val):.4f}")
                else:
                    print(f"    ← {r:<24s} value={val}")
            else:
                print(f"    ← {r:<24s} *** MISSING ***")

        residual_before = float(np.linalg.norm(state.current_target))

        # Apply
        state = token.apply(state)

        residual_after = float(np.linalg.norm(state.current_target))

        # Show what was written
        print(f"\n  WROTE to state.features:")
        for w in token.writes:
            if w in state.features:
                val = state.features[w]
                if hasattr(val, 'shape'):
                    print(f"    → {w:<24s} shape={val.shape}  mean={np.nanmean(val):.4f}")
                else:
                    print(f"    → {w:<24s} value={val}")

        print(f"\n  Residual norm: {residual_before:.4f} → {residual_after:.4f}  "
              f"(change: {residual_after - residual_before:+.4f})")

        describe_state(state, f"Step {step}: {name}", token=token)

    # ── Compact summary trace ──────────────────────────────────
    print(f"\n{'=' * 90}")
    print(f"  TRANSFORMATION LOG (from state.log)")
    print(f"{'=' * 90}")
    state.print_log()

    return state


replayed_state = replay_detailed(TOKEN_SEQUENCE, H, F, replay_grammar)


### Forecast visualization

In [ ]:
forecast = replayed_state.features.get("final_forecast")
if forecast is not None:
    n_show = min(4, H.shape[0])
    fig, axes = plt.subplots(n_show, 1, figsize=(12, 3 * n_show), sharex=False)
    if n_show == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        tail = 120
        ax.plot(np.arange(-tail, 0), H[i, -tail:], label="history (last 120)", lw=0.8)
        ax.plot(np.arange(F.shape[1]), F[i], label="ground truth", lw=0.8, color="tab:green")
        ax.plot(np.arange(F.shape[1]), forecast[i], label="CLP forecast", lw=0.8, ls="--", color="tab:red")
        ax.axvline(0, ls=":", color="grey", alpha=0.6)
        ax.set_title(f"Sample {i}  —  MASE = {replayed_state.mase:.4f}" if replayed_state.mase else f"Sample {i}")
        ax.legend(fontsize=8); ax.grid(alpha=0.25)
    plt.tight_layout(); plt.show()
else:
    print("No final_forecast found in state — did the sequence include STOP?")

print(f"\nFinal MASE: {replayed_state.mase}")


## 4. Grammar Details

The Grammar is a directed graph where nodes are token names (plus `START`)
and edges encode which token may follow which. Beyond the graph structure,
`valid_actions()` also checks that the required feature keys are present
in the state — so even if the graph edge exists, a token is only truly valid
if its `reads` keys are available.


In [ ]:
print(g)
print()
print("Valid at START (fresh state):", g.valid_actions(State(H, F)))

st = State(H, F)
CleanNormalize().apply(st)
print("Valid after 'normalize'     :", g.valid_actions(st))

FeatRaw().apply(st)
print("Valid after 'feat_raw'      :", g.valid_actions(st))

ModelKernelRBF().apply(st)
print("Valid after 'kernel_rbf'    :", g.valid_actions(st))


In [ ]:
print(inspect.getsource(Grammar.valid_actions))


## 5. Heuristics — MI Score and Action Priors

The mutual-information score guides MCTS by estimating which model tokens
are likely to reduce the residual the most. `compute_action_priors` turns
that into a probability distribution over candidate actions.


In [ ]:
st = State(H, F)
CleanNormalize().apply(st)
FeatLagFeatures().apply(st)

mi = compute_mi_score(st)
actions = ["kernel_rbf", "random_forest", "xgboost", "STOP"]
print("MI score:", mi)
print()
print("Action priors:")
pprint(compute_action_priors(mi, actions))


## 6. MCTS Search

Grammar-guided Monte Carlo Tree Search with PUCT exploration.
Reward = 1 / (1 + MASE).


In [ ]:
RUN_MCTS = True

if RUN_MCTS:
    res = mcts_search(g, State(H[:8], F[:8]), n_iterations=5, verbose=True)
    print("\nBest chain:", res["best_chain"])
    print("Best MASE :", res["best_mase"])
    print_mcts_tree(res["root"], max_depth=4)


## 7. Exhaustive Search over Short Chains

While the grammar is small we can enumerate all cleaning → feature → model(s) chains
and rank them by MASE on the electricity data.


In [ ]:
from itertools import product

def run_chain(chain, H=H, F=F):
    st = State(H, F)
    for name in chain:
        g.tokens[name].apply(st)
    if not st.terminated:
        g.tokens["STOP"].apply(st)
    return st

chains = []
for cleaner in ["normalize", "detrend_norm"]:
    for feature in ["feat_raw", "fft_encode", "feat_lag"]:
        for depth in [1, 2]:
            for models in product(["kernel_rbf"], repeat=depth):
                chains.append([cleaner, feature, *models, "STOP"])

rows = []
for chain in chains:
    try:
        st = run_chain(chain, H[:8], F[:8])
        rows.append((st.mase, " → ".join(chain)))
    except Exception as exc:
        rows.append((np.inf, " → ".join(chain) + " ERROR " + repr(exc)))

print("Top 10 chains by MASE (lower is better):")
for i, (mv, chain) in enumerate(sorted(rows)[:10], 1):
    print(f"  {i:2d}. MASE={mv:8.4f} | {chain}")


## 8. Residual Chaining Demo

Model 2 receives the residual left by model 1. `State.push_prediction()` keeps
the residual in the active target scale, so normalized chains do not mix
normalized predictions with raw futures.


In [ ]:
st = State(H, F)
CleanNormalize().apply(st)
FeatLagFeatures().apply(st)

print("After normalize + feat_lag:")
print(f"  target norm = {np.linalg.norm(st.current_target):.4f}")

ModelKernelRBF().apply(st)
print(f"\nAfter model 1 (kernel_rbf):")
print(f"  target norm = {np.linalg.norm(st.current_target):.4f}")
print(f"  predictions = {st.prediction_names}")

ModelKernelRBF().apply(st)
print(f"\nAfter model 2 (kernel_rbf on residual):")
print(f"  target norm = {np.linalg.norm(st.current_target):.4f}")
print(f"  predictions = {st.prediction_names}")

StopToken().apply(st)
print(f"\nAfter STOP:")
print(f"  prediction stack: {st.prediction_names}")
print(f"  MASE = {st.mase}")


## 9. Leakage Checklist

- Feature tokens should not read `future`, `original_future`, `current_target`, or residuals.
- Forecasting augmentation should not move future values into history.
- If targets are normalized, residual updates must remain normalized until STOP.
- STOP is the only place that should compare forecast to ground truth.
- LOO models must not train on the held-out sample.
- Augmented copies can make LOO optimistic if near-duplicates remain in the training fold.
  For final evaluation, split originals first, augment training only, evaluate on untouched originals.


## 10. Where to Add New Ideas

- **New cleaner**: `tokens/cleaning.py`, then `tokens/__init__.py`.
- **New feature extractor**: `tokens/features.py`.
- **New model/residual learner**: `tokens/models.py`.
- **New search prior**: `heuristics.py`.
- **New grammar policy**: `register_all` or a notebook-local grammar builder.
- **New low-level kernel**: `kernels.py`.
- **New decomposition feature**: `decomposition.py` or a token that wraps it.
